# NB-18 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-18**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-18.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This extends the bid-book decision problem to beginning inventory and overlapping deliveries. It uses the approved deterministic holdings and funding assumptions with retained market and award arrays.

- [Volume VI, Chapter 12 — The submitted FTR bid book](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c12)
- [Volume VI, Chapter 10 — Settlement as a network target](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c10)
- [Volume VI, Chapter 13 — Real-data benchmark and implementation gaps](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c13)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-18', require_key=True)
print(setup_status)


# NB-18 — Existing inventory, funding and overlapping delivery books
Companion to Volume VI §§10,12–13; prerequisites NB-17 and NB-16.
This is the user-approved **new bounded controlled extension**. Market and
auction arrays are retained; beginning holdings, delivery offsets, fees,
collateral and funding stresses are new declared assumptions. No random market
observations or award scenarios are generated.

Compare a finite menu of buy requests under uncertain awards. Sales are not
allowed. Risk is scored on owned plus awarded exposure, never only new trades.
This is not a calibrated desk model or a reconstruction of an actual auction.

In [ ]:
NOTEBOOK_ID='NB-18'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

raw_inspection={k:np.asarray(v) for k,v in inputs['raw_inspection'].items()}
entry=inputs['requests']['complete_inventory']
response=execute('complete_inventory',entry['method'],entry['request'])
inventory=response['inventory_decision_result']
cfg=inventory['config'];results=inventory['results'];support=inventory['support']
details={case:{k:np.asarray(v) for k,v in values.items()} for case,values in inventory['details'].items()}
plt=setup_plot()
print(json.dumps(cfg,indent=2));print('Retained support:',support)

## Owned, requested and awarded are separate ledgers
Beginning inventory is 5 MW of ALL_N3_N10 for prompt days [0,30). New requests
are for ALL_N3_N10 and ALL_N7_N10 on forward days [15,45). Each path may request
0,5,10 MW; owned plus requested MW cannot exceed 15. No sale or automatic exit
is permitted. A desired final book minus x₀ is not an executable order.

Bid limits are fixed from training forward payouts: max(mean−0.22 ES₉₅(loss),0).
The original auction LP clears requests at each retained price and residual
corridor. During the 15-day overlap, existing exposure is subtracted from that
corridor; outside overlap only new exposure applies. The LP uses the tighter
bound from these periods. An existing book that violates this declared corridor
fails rather than being silently liquidated.

This explicitly assumes the retained 30-day auction prices and residual
corridors can be transferred to the forward product. It is a controlled
stationarity assumption, not an empirical auction-calendar alignment.

In [ ]:
for name,r in results.items():print(name,'old:',r['old'],'selected:',r['best'])
fig,ax=plt.subplots(figsize=(10,4));ax.broken_barh([(0,30)],(1.1,.6),facecolors='#88C0D0');ax.broken_barh([(15,30)],(.1,.6),facecolors='#B48EAD');ax.axvspan(15,30,color='#EBCB8B',alpha=.15,label='Shared exposure days')
ax.set(xlim=(0,45),ylim=(0,2),yticks=[.4,1.4],yticklabels=['New forward [15,45)','Owned prompt [0,30)'],xlabel='Relative delivery day · all 24 hourly intervals',title='Two products on one delivery calendar');ax.legend();plt.show()

## Joint delivery scenarios preserve the overlap
The original archive holds overlapping 30-day payout totals. Pair row k with
row k+15, preserving the 15 shared delivery days on the same market path.
There are 206 training pairs and 71 held-out pairs; they overlap and are not
independent delivery experiments. Pairing occurs wholly within each split.

Every retained auction scenario is crossed with every paired delivery scenario
with uniform weight. Auction/settlement independence is a declared assumption,
not learned joint dependence. The 24 auctions do not manufacture independent
market years. Optimize the finite request menu on training rows only; held-out
rows merely score that selected request. Empty and inventory cases use exactly
the same forecasts, prices and decision rule.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4));names=list(results)
for j,name in enumerate(names):
    r=results[name];axes[0].bar(j-.18,sum(r['best']['requested']),.36,color='#88C0D0');axes[0].bar(j+.18,sum(r['best']['mean_awarded']),.36,color='#B48EAD')
axes[0].set(xticks=range(4),xticklabels=names,ylabel='New MW',title='Requested (blue) versus mean awarded (purple)');axes[0].tick_params(axis='x',rotation=15)
for name,color in [('empty','#88C0D0'),('inventory','#B48EAD')]:
    curve=sorted([r for r in results[name]['candidates'] if r['requested'][0]==0],key=lambda r:r['requested'][1]);zero=curve[0]['objective']
    axes[1].plot([r['requested'][1] for r in curve],[r['objective']-zero for r in curve],marker='o',label=name,color=color)
    print(name,'conditional request-value increments per 5 requested MW:',np.diff([r['objective'] for r in curve]).tolist())
axes[1].set(xlabel='Requested MW on ALL_N7_N10 (first request zero)',ylabel='Training objective gain from no new trade ($)',title='Finite-step marginal request value');axes[1].legend();fig.tight_layout();plt.show()

## Cash expenses, funding and collateral are different objects
Forward cash = gross owned settlement + gross awarded settlement + funding
adjustments − signed acquisition price × awarded MW − fees. Original acquisition
cost of owned rights is sunk and excluded from this incremental forward ledger;
lifetime accounting would add that historical cash entry once.

Negative acquisition cost is a cash credit, not a sign error. Fees apply once
to awarded new MW. The funding stress pays 50% of positive **instrument-product**
settlement credits and 100% of negative obligations; it applies to both owned
and new rights before aggregation. This asymmetric rule can worsen a hedge.

Reserve $2,000 per owned-or-requested MW against a $30,000 collateral budget.
Collateral is a liquidity constraint, not a P&L expense. Release timing, interest,
default and daily mark-to-market are not modeled. The no-exit assumption means
owned exposure remains even when the best incremental action is no trade.

In [ ]:
for name,r in results.items():
    b=r['best'];assert np.isclose(b['mean_test_net'],b['mean_gross']+b['mean_funding_adjustment']-b['mean_acquisition']-b['mean_fees'])
    print(name,{k:b[k] for k in ['mean_gross','mean_funding_adjustment','mean_acquisition','mean_fees','mean_test_net','collateral_reserved','test_es95_loss']})
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(names,[results[n]['best']['mean_test_net'] for n in names],color=['#88C0D0','#B48EAD','#D08770','#A3BE8C']);axes[0].axhline(0,color='#ECEFF4');axes[0].set(ylabel='Held-out mean forward cash ($)',title='Full holdings ledger');axes[0].tick_params(axis='x',rotation=15)
axes[1].bar(names,[results[n]['best']['test_es95_loss'] for n in names],color=['#88C0D0','#B48EAD','#D08770','#A3BE8C']);axes[1].set(ylabel='Held-out ES95 loss ($)',title='Fractional empirical tail mass on total holdings');axes[1].tick_params(axis='x',rotation=15);fig.tight_layout();plt.show()

## What changed, what did not, and the no-trade control
On this tiny retained menu the empty and inventory cases select the same new
request. Inventory still changes the cash and tail risk. Do not force a different
trade to make the example look successful. The funding stress changes sizing;
the deliberately extreme $100,000-per-awarded-MW fee stress selects no new trade.
That fee is a failure control, not a realistic fee estimate. No new trade does
not mean zero risk: the owned book is still exposed and cannot be sold here.

The finite-step request-value curves are not globally optimal bid curves: bid
limits are fixed, support is only two paths, and quantities lie on a coarse
grid. No confidence interval is inferred from overlapping scenario rows.
Real deployment needs observed holdings, contract calendars, clearing/funding
rules, liquidity and exit mechanics, and a validated joint scenario model.

In [ ]:
assert results['fee_stress']['best']['requested']==[0.,0.]
assert results['fee_stress']['best']['test_es95_loss']>0
assert results['empty']['best']['requested']==results['inventory']['best']['requested']

for control in inputs['negative_controls']:reject(control['method'],control['request'])
checked=dict(config=cfg,results=results,support=support,details={case:{k:v.tolist() for k,v in values.items()} for case,values in details.items()})
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(complete_retained_inventory=True,original_approved_assumptions_unchanged=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)